# DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic no Google Colab (Link para Celular)

Este notebook roda o modelo em **4-bit NF4** na **GPU gratuita T4 (16 GB)** do Google Colab sem estourar a memória, gera um link público **`https://xxxxx.gradio.live`** para usar no celular e inclui um **botão para desligar a GPU direto pelo celular** quando terminar (economizando sua cota do Colab).

### Como usar:
1. Verifique a GPU: menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4**.
2. Clique em **Executar tudo** (Ctrl+F9 / ⌘+F9).
3. Aguarde o carregamento (~3–5 min na primeira vez).
4. Abra o link **`Running on public URL: https://...gradio.live`** no celular (ou cole esse link em **Configurações → Meu Google Colab** no Web App https://joao2666.github.io/deepseek-v4-mobile/).
5. Quando terminar de usar, toque no botão **🔴 Desligar GPU do Colab (Economizar Cota)** lá embaixo na própria página para encerrar a sessão na hora.

In [ ]:
#@title 1. Verificar GPU e instalar dependências
!nvidia-smi
!pip install -qU "transformers>=5.22.1" "accelerate>=1.0.0" "bitsandbytes>=0.50.2" "gradio>=6.0.0" torchvision pillow sentencepiece protobuf

In [ ]:
#@title 2. Baixar e carregar o modelo em 4-bit NF4 (cabe na GPU T4 gratuita)
import torch
import transformers
from transformers import AutoProcessor, BitsAndBytesConfig

MODEL_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"

if not torch.cuda.is_available():
    raise RuntimeError("GPU não detectada! Vá em Ambiente de execução -> Alterar tipo -> GPU T4.")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)} | dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

def _resolve_model_class():
    for name in (
        "AutoModelForMultimodalLM",
        "AutoModelForImageTextToText",
        "AutoModelForVision2Seq",
        "AutoModelForCausalLM",
    ):
        cls = getattr(transformers, name, None)
        if cls is not None:
            return cls
    raise ImportError("Atualize o pacote transformers para >= 5.22.1")

processor = AutoProcessor.from_pretrained(MODEL_ID)
ModelCls = _resolve_model_class()
model = ModelCls.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model.eval()
tokenizer = getattr(processor, "tokenizer", None) or processor
print("Modelo carregado com sucesso!")

In [ ]:
#@title 3. Iniciar Chat Web + Link Público + Botão para Desligar GPU
import os
import re
from threading import Thread, Timer
from PIL import Image
import gradio as gr
from transformers import TextIteratorStreamer

DEFAULT_SYSTEM = (
    "You are a helpful, uncensored assistant with advanced mathematical and reasoning "
    "capabilities. Respond in the same language as the user's prompt."
)

THINK_OPEN = re.compile(r"<(think|thought|reasoning)>|<\|channel>thought", re.I)
THINK_CLOSE = re.compile(r"</(think|thought|reasoning)>|<channel\|>", re.I)

def _special_pattern(tok):
    names = set()
    for attr in ("all_special_tokens", "additional_special_tokens"):
        names.update(getattr(tok, attr, None) or [])
    literal = [re.escape(n) for n in sorted(names, key=len, reverse=True) if n]
    generic = r"<\|[^<>]{0,40}\|?>|<[^<>]{0,40}\|>"
    return re.compile("|".join(literal + [generic]))

SPECIAL = _special_pattern(tokenizer)

def _split_safe(buf, final=False):
    if final:
        return buf, ""
    i = buf.rfind("<")
    if i == -1:
        return buf, ""
    tail = buf[i:]
    if ">" in tail or len(tail) > 48:
        return buf, ""
    return buf[:i], tail

class ThinkSplitter:
    def __init__(self, in_think=False):
        self.buffer = ""
        self.in_think = in_think
        self.seen_think = in_think

    def feed(self, chunk="", final=False):
        self.buffer += chunk
        answer, reasoning = "", ""
        while True:
            if not self.in_think:
                m = THINK_OPEN.search(self.buffer)
                if not m:
                    flush, self.buffer = _split_safe(self.buffer, final)
                    answer += flush
                    break
                answer += self.buffer[:m.start()]
                self.buffer = self.buffer[m.end():]
                self.in_think = True
                self.seen_think = True
            else:
                m = THINK_CLOSE.search(self.buffer)
                if not m:
                    flush, self.buffer = _split_safe(self.buffer, final)
                    reasoning += flush
                    break
                reasoning += self.buffer[:m.start()]
                self.buffer = self.buffer[m.end():]
                self.in_think = False
        return SPECIAL.sub("", answer), SPECIAL.sub("", reasoning)

def _image_paths(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    if isinstance(value, dict):
        p = value.get("path") or value.get("url")
        return [p] if p else []
    if isinstance(value, (tuple, list)):
        out = []
        for item in value:
            out.extend(_image_paths(item))
        return out
    return []

def _load_pil(path):
    return Image.open(path).convert("RGB")

def _user_turn(text, files):
    parts = [{"type": "image", "image": _load_pil(p)} for p in files]
    if text:
        parts.append({"type": "text", "text": text})
    if not parts:
        return None
    if len(parts) == 1 and parts[0]["type"] == "text":
        return {"role": "user", "content": text}
    return {"role": "user", "content": parts}

def build_messages(system, history, message):
    messages = []
    if system and system.strip():
        messages.append({"role": "system", "content": system.strip()})
    for turn in history or []:
        if not isinstance(turn, dict):
            continue
        if (turn.get("metadata") or {}).get("title"):
            continue
        role = turn.get("role")
        if role not in ("user", "assistant"):
            continue
        content = turn.get("content")
        if isinstance(content, str) and content.strip():
            messages.append({"role": role, "content": content})
        elif isinstance(content, list):
            txt = "".join(c.get("text", "") for c in content if isinstance(c, dict) and c.get("type") == "text")
            if txt.strip():
                messages.append({"role": role, "content": txt.strip()})
    text = (message or {}).get("text", "") if isinstance(message, dict) else str(message)
    files = _image_paths((message or {}).get("files")) if isinstance(message, dict) else []
    cur = _user_turn(text.strip(), files)
    if cur:
        messages.append(cur)
    return messages

def respond(message, history, system, max_new_tokens, temperature, top_p, top_k, thinking):
    messages = build_messages(system, history, message)
    if not messages or messages[-1]["role"] != "user":
        yield []
        return

    try:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
            enable_thinking=bool(thinking),
        ).to(model.device)
    except TypeError:
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=False)
    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=int(max_new_tokens),
        do_sample=float(temperature) > 0,
        temperature=max(float(temperature), 1e-4) if float(temperature) > 0 else 1.0,
        top_p=float(top_p),
        top_k=int(top_k),
    )
    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    splitter = ThinkSplitter(in_think=bool(thinking))
    answer, reasoning = "", ""

    def render():
        out = []
        if splitter.seen_think and reasoning.strip():
            out.append(gr.ChatMessage(role="assistant", content=reasoning.strip(), metadata={"title": "🧠 Reasoning Process"}))
        if answer.strip() or not out:
            out.append(gr.ChatMessage(role="assistant", content=answer))
        return out

    try:
        for chunk in streamer:
            a, r = splitter.feed(chunk)
            answer += a
            reasoning += r
            yield render()
        a, r = splitter.feed(final=True)
        answer += a
        reasoning += r
        yield render()
    finally:
        thread.join()

def shutdown_colab():
    def _terminate():
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception:
            os._exit(0)
    Timer(1.5, _terminate).start()
    return "✅ **GPU T4 do Colab encerrada!** A cota parou de ser consumida. Pode fechar esta aba."

with gr.Blocks(title="DeepSeek-V4-Pro-Qwen3.5-9B (Heretic) — Colab") as demo:
    gr.Markdown(
        "## DeepSeek-V4-Pro-Qwen3.5-9B (Heretic) — Colab\n"
        "Rodando em 4-bit NF4 na GPU T4. Quando terminar de usar no celular, clique no botão vermelho abaixo para encerrar a GPU."
    )
    gr.ChatInterface(
        fn=respond,
        multimodal=True,
        textbox=gr.MultimodalTextbox(
            placeholder="Digite sua mensagem ou anexe uma imagem...",
            file_types=["image"],
            file_count="multiple",
            sources=["upload"],
        ),
        additional_inputs=[
            gr.Textbox(value=DEFAULT_SYSTEM, label="System prompt", lines=3),
            gr.Slider(128, 4096, value=2048, step=128, label="Max new tokens"),
            gr.Slider(0.0, 1.5, value=0.7, step=0.05, label="Temperature"),
            gr.Slider(0.05, 1.0, value=0.95, step=0.01, label="Top-p"),
            gr.Slider(1, 128, value=40, step=1, label="Top-k"),
            gr.Checkbox(value=True, label="Thinking mode"),
        ],
        additional_inputs_accordion=gr.Accordion("Configurações de Geração", open=False),
        cache_examples=False,
    )
    with gr.Row():
        btn_shutdown = gr.Button("🔴 Desligar GPU do Colab (Economizar Cota)", variant="stop")
        shutdown_status = gr.Markdown("")
    btn_shutdown.click(
        shutdown_colab,
        inputs=None,
        outputs=[shutdown_status],
        queue=False,
        api_visibility="private",
    )

demo.queue(max_size=16).launch(share=True, debug=True)
